# LangChain 1.3.4 + FastMCP + `create_agent()` Example

This notebook demonstrates how to use **LangChain 1.3.4** with a **FastMCP** server exposing cybersecurity tools.

The referenced MCP server exposes five tools:

- `ask_cis`
- `lookup_cve`
- `lookup_cisa_kev`
- `lookup_attack`
- `lookup_epss`

Start the MCP server before running the agent example.


## Project Structure

```text
project/
│
├── mcp_cyber_tools_server.py
├── agent.py
├── .env
└── requirements.txt
```


## requirements.txt

```text
langchain==1.3.4
langchain-openai
langchain-mcp-adapters
mcp
fastmcp
python-dotenv
```


## Start the MCP Server

```bash
python mcp_cyber_tools_server.py
```


## Complete `agent.py`

In [2]:
!pip install langchain-mcp-adapters

In [1]:
import asyncio
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

async def main():

    client = MultiServerMCPClient(
        {
            "cyber": {
                "transport": "stdio",
                "command": "python",
                "args": ["mcp_cyber_tools_server.py"],
            }
        }
    )

    tools = await client.get_tools()

    print("\nAvailable Tools\n----------------")
    for t in tools:
        print(t.name)

    llm = ChatOpenAI(
        model="gpt-4.1",
        temperature=0,
    )

    agent = create_agent(
        model=llm,
        tools=tools,
    )

    response = await agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "Find recent Apache Struts CVEs and "
                        "also retrieve their EPSS score."
                    ),
                }
            ]
        }
    )

    print("\n===========================")
    print(response["messages"][-1].content)

if __name__ == "__main__":
    asyncio.run(main())


RuntimeError: asyncio.run() cannot be called from a running event loop

## What Happens

```
User
    │
    ▼
create_agent()

    │

    ├── lookup_cve("Apache Struts")

    │
    ▼
Returns

CVE-2024-...
CVE-2023-...

    │

    ├── lookup_epss("CVE-2024-...")
    ├── lookup_epss("CVE-2023-...")

    │
    ▼

Final Answer
```

No routing code is required.


## Example 2

```python
response = await agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '''
Search MITRE ATT&CK for Kerberoasting.
Then search CISA KEV for Windows Print Spooler.
'''
            }
        ]
    }
)
```

The agent automatically calls:

```text
lookup_attack()
lookup_cisa_kev()
```


## Example 3 - CIS RAG

```python
response = await agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "According to CIS Windows 11 benchmark, how do I disable SMBv1?"
            }
        ]
    }
)
```

The agent chooses:

```text
ask_cis(
    "According to CIS Windows 11 benchmark, how do I disable SMBv1?"
)
```


## Display Available MCP Tools

```python
tools = await client.get_tools()

for tool in tools:
    print(tool.name)
    print(tool.description)
```

Expected:

```text
ask_cis
lookup_cve
lookup_cisa_kev
lookup_attack
lookup_epss
```


## Architecture

```text
           create_agent()
                  │
                  ▼
      MultiServerMCPClient
                  │
                  ▼
      FastMCP Cyber Server
                  │
        ┌─────────┼─────────┐
        │         │         │
   ask_cis   lookup_cve  lookup_epss
        │         │         │
        └─────────┼─────────┘
                  ▼
             Final Response
```

This notebook preserves the code and explanation from the original response.
